# LiSSA on Colab (L4): RQ2 and RQ3, two passes on the dev slice

Every session, including after a disconnect: an **L4 GPU** runtime, cells 1 to 4 in order, then the pipeline cell. Finished steps are skipped and saved answers are reused.

**Pass 1:** the baseline and the four prompt variants for both models; claim verification of the baseline and the best variant; the choice of τ.

**Pass 2, decided before pass 1's results are known:** combine every information type that beat the baseline on dev (or the two best if fewer did), dropping the weakest part if the prompt would not fit the context window. Then verify the best prompt so far.

Only the dev slice is used. The test slice is run once, after the choices are frozen.

**Cell 1: connect Google Drive and check the GPU** (it stops if the GPU is not an L4, so every run uses the same GPU type)

In [1]:
from google.colab import drive
drive.mount("/content/drive")
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)
assert "L4" in gpu, "Not an L4. Change the runtime type to L4, then start again."

Mounted at /content/drive
NVIDIA L4, 23034 MiB


**Cell 2: install everything (about 3 to 5 minutes)**

In [2]:
%%bash
cd /content
[ -d LiSSA-Replication-Package ] || git clone -q --depth 1 -b rq2-rq3-harness https://github.com/rahulxdagar/LiSSA-Replication-Package.git
[ -d Replication-Package-ICSE25_LiSSA-Toward-Generic-Traceability-Link-Recovery-through-RAG ] || git clone -q --depth 1 https://github.com/ardoco/Replication-Package-ICSE25_LiSSA-Toward-Generic-Traceability-Link-Recovery-through-RAG.git
rm -f LiSSA-Replication-Package/harness/data/candidates-local.json

apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y openjdk-21-jdk-headless maven zstd rsync > /dev/null 2>&1
printf '#!/bin/sh\necho /usr/lib/jvm/java-21-openjdk-amd64\n' > /usr/libexec/java_home && chmod +x /usr/libexec/java_home

curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.2 sh > /dev/null
echo 'IiIiUHJvbXB0IHZhcmlhbnRzIGZvciBSUTIuIEEgdmFyaWFudCB0dXJucyBvbmUgY2FuZGlkYXRlIHBhaXIgaW50byAoc3lzdGVtIG1lc3NhZ2UsIHVzZXIgbWVzc2FnZSkuCgonYmFzZWxpbmUnIGlzIExpU1NBJ3MgY2hhaW4tb2YtdGhvdWdodCBwcm9tcHQsIGJ5dGUgZm9yIGJ5dGUsIHNvIGl0cyBjYWNoZSBrZXlzIG1hdGNoIFJRMSdzLiBFYWNoIFJRMiB2YXJpYW50IGFkZHMgb25lCnBpZWNlIG9mIGluZm9ybWF0aW9uIGFuZCBjaGFuZ2VzIG5vdGhpbmcgZWxzZTogTGlTU0EncyBzeXN0ZW0gbWVzc2FnZSwgcXVlc3Rpb24sIGFuc3dlciBmb3JtYXQgYW5kIGFydGlmYWN0cyBzdGF5CmV4YWN0bHkgYXMgdGhleSBhcmUuIFRoZSBhZGRlZCB0ZXh0cyBjb21lIGZyb20gcHJvbXB0cy8gYW5kIGRhdGEvIChzZWUgcHJvbXB0cy9SRUFETUUubWQpLgoKICBkZWZpbml0aW9uICBhIHByZWNpc2UgZGVmaW5pdGlvbiBvZiBhIHRyYWNlIGxpbmssIGFmdGVyIHRoZSBmaXJzdCBzZW50ZW5jZQogIHByb2plY3QgICAgIGEgc2hvcnQgZGVzY3JpcHRpb24gb2YgdGhlIHN5c3RlbSwgYWZ0ZXIgdGhlIGZpcnN0IHNlbnRlbmNlCiAgZXhhbXBsZXMgICAgdHdvIGxhYmVsbGVkIHBhaXJzIGZyb20gdGhlIG90aGVyIHByb2plY3RzLCBhZnRlciB0aGUgZmlyc3Qgc2VudGVuY2UKICBzdW1tYXJ5ICAgICBhIHN0cnVjdHVyYWwgc3VtbWFyeSBvZiB0aGUgY29kZSBmaWxlICh0eXBlcywgZmllbGRzLCBzaWduYXR1cmVzLCBKYXZhZG9jKSwgYWZ0ZXIgdGhlIGNvZGUKIiIiCgppbXBvcnQgZnVuY3Rvb2xzCmltcG9ydCBqc29uCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwoKaW1wb3J0IGxpc3NhX2NvbXBhdCBhcyBsYwoKSEVBRCA9ICJCZWxvdyBhcmUgdHdvIGFydGlmYWN0cyBmcm9tIHRoZSBzYW1lIHNvZnR3YXJlIHN5c3RlbS4iClFVRVNUSU9OID0gbGMuQ09UX1BST01QVFtsZW4oSEVBRCkgKyAxOl0gICMgIklzIHRoZXJlIGEgdHJhY2VhYmlsaXR5IGxpbmsgYmV0d2VlbiAoMSkgYW5kICgyKT8gLi4uICcnJ3t0YXJnZXRfY29udGVudH0nJycgIgphc3NlcnQgbGMuQ09UX1BST01QVC5zdGFydHN3aXRoKEhFQUQgKyAiICIpClBST01QVFMgPSBsYy5IQVJORVNTIC8gInByb21wdHMiCkRBVEEgPSBsYy5IQVJORVNTIC8gImRhdGEiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUGFpcjoKICAgIHByb2plY3Q6IHN0cgogICAgcmVxdWlyZW1lbnQ6IHN0ciAgIyBmaWxlIG5hbWUsIGUuZy4gIlVDMS50eHQiCiAgICBjb2RlOiBzdHIgICMgZmlsZSBuYW1lLCBlLmcuICJEQkN1bHR1cmFsSGVyaXRhZ2UuamF2YSIKICAgIHJlcXVpcmVtZW50X3RleHQ6IHN0cgogICAgY29kZV90ZXh0OiBzdHIKCgpAZnVuY3Rvb2xzLmNhY2hlCmRlZiBwcm9tcHRfdGV4dChuYW1lKToKICAgIHJldHVybiAoUFJPTVBUUyAvIG5hbWUpLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKS5yc3RyaXAoIlxuIikKCgpAZnVuY3Rvb2xzLmNhY2hlCmRlZiBzdW1tYXJpZXMocHJvamVjdCk6CiAgICByZXR1cm4ganNvbi5sb2FkcygoREFUQSAvIGYic3VtbWFyaWVzLXtwcm9qZWN0fS5qc29uIikucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpKQoKCmRlZiBfd2l0aF9pbmZvcm1hdGlvbihsYWJlbCwgaW5mb3JtYXRpb24sIHBhaXIpOgogICAgcXVlc3Rpb24gPSBsYy5maWxsKFFVRVNUSU9OLCBwYWlyLnJlcXVpcmVtZW50X3RleHQsIHBhaXIuY29kZV90ZXh0KQogICAgcmV0dXJuIGxjLlNZU1RFTV9NRVNTQUdFLCBmIntIRUFEfVxue2xhYmVsfToge2luZm9ybWF0aW9ufVxue3F1ZXN0aW9ufSIKCgpkZWYgYmFzZWxpbmUocGFpcik6CiAgICByZXR1cm4gbGMuU1lTVEVNX01FU1NBR0UsIGxjLmZpbGwobGMuQ09UX1BST01QVCwgcGFpci5yZXF1aXJlbWVudF90ZXh0LCBwYWlyLmNvZGVfdGV4dCkKCgpkZWYgZGVmaW5pdGlvbihwYWlyKToKICAgIHJldHVybiBfd2l0aF9pbmZvcm1hdGlvbigiRGVmaW5pdGlvbiIsIHByb21wdF90ZXh0KCJkZWZpbml0aW9uLnR4dCIpLCBwYWlyKQoKCmRlZiBwcm9qZWN0KHBhaXIpOgogICAgcmV0dXJuIF93aXRoX2luZm9ybWF0aW9uKCJBYm91dCB0aGlzIHN5c3RlbSIsIHByb21wdF90ZXh0KGYicHJvamVjdC17cGFpci5wcm9qZWN0fS50eHQiKSwgcGFpcikKCgpkZWYgZXhhbXBsZXMocGFpcik6CiAgICByZXR1cm4gX3dpdGhfaW5mb3JtYXRpb24oIkxhYmVsbGVkIGV4YW1wbGVzIGZyb20gb3RoZXIgc29mdHdhcmUgc3lzdGVtcyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIlxuIiArIHByb21wdF90ZXh0KGYiZXhhbXBsZXMte3BhaXIucHJvamVjdH0udHh0IiksIHBhaXIpCgoKZGVmIHN1bW1hcnkocGFpcik6CiAgICBzeXN0ZW0sIHVzZXIgPSBiYXNlbGluZShwYWlyKQogICAgcmV0dXJuIHN5c3RlbSwgZiJ7dXNlcn1cbiBTdHJ1Y3R1cmFsIHN1bW1hcnkgb2YgKDIpLCBleHRyYWN0ZWQgYnkgYSBwYXJzZXI6ICcnJ3tzdW1tYXJpZXMocGFpci5wcm9qZWN0KVtwYWlyLmNvZGVdfScnJyAiCgoKQkFTRV9WQVJJQU5UUyA9IHsiYmFzZWxpbmUiOiBiYXNlbGluZSwgImRlZmluaXRpb24iOiBkZWZpbml0aW9uLCAic3VtbWFyeSI6IHN1bW1hcnksICJwcm9qZWN0IjogcHJvamVjdCwKICAgICAgICAgICAgICAgICAiZXhhbXBsZXMiOiBleGFtcGxlc30KCiMgUm91bmQgMjogc2V2ZXJhbCBraW5kcyBvZiBpbmZvcm1hdGlvbiBpbiBvbmUgcHJvbXB0LCBpbiB0aGUgb3JkZXIgb2YgdGhlIHNpbmdsZSB2YXJpYW50cwpQQVJUUyA9ICgiZGVmaW5pdGlvbiIsICJwcm9qZWN0IiwgImV4YW1wbGVzIiwgInN1bW1hcnkiKQpfSU5GT1JNQVRJT04gPSB7CiAgICAiZGVmaW5pdGlvbiI6IGxhbWJkYSBwYWlyOiAoIkRlZmluaXRpb24iLCBwcm9tcHRfdGV4dCgiZGVmaW5pdGlvbi50eHQiKSksCiAgICAicHJvamVjdCI6IGxhbWJkYSBwYWlyOiAoIkFib3V0IHRoaXMgc3lzdGVtIiwgcHJvbXB0X3RleHQoZiJwcm9qZWN0LXtwYWlyLnByb2plY3R9LnR4dCIpKSwKICAgICJleGFtcGxlcyI6IGxhbWJkYSBwYWlyOiAoIkxhYmVsbGVkIGV4YW1wbGVzIGZyb20gb3RoZXIgc29mdHdhcmUgc3lzdGVtcyIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJcbiIgKyBwcm9tcHRfdGV4dChmImV4YW1wbGVzLXtwYWlyLnByb2plY3R9LnR4dCIpKSwKfQoKCmRlZiBjb21iaW5lZChwYXJ0cyk6CiAgICBkZWYgdmFyaWFudChwYWlyKToKICAgICAgICBxdWVzdGlvbiA9IGxjLmZpbGwoUVVFU1RJT04sIHBhaXIucmVxdWlyZW1lbnRfdGV4dCwgcGFpci5jb2RlX3RleHQpCiAgICAgICAgYmxvY2tzID0gIiIuam9pbihmIntsYWJlbH06IHt0ZXh0fVxuIiBmb3IgbGFiZWwsIHRleHQgaW4KICAgICAgICAgICAgICAgICAgICAgICAgIChfSU5GT1JNQVRJT05bcF0ocGFpcikgZm9yIHAgaW4gUEFSVFMgaWYgcCBpbiBwYXJ0cyBhbmQgcCBpbiBfSU5GT1JNQVRJT04pKQogICAgICAgIHVzZXIgPSBmIntIRUFEfVxue2Jsb2Nrc317cXVlc3Rpb259IgogICAgICAgIGlmICJzdW1tYXJ5IiBpbiBwYXJ0czoKICAgICAgICAgICAgdXNlciArPSBmIlxuIFN0cnVjdHVyYWwgc3VtbWFyeSBvZiAoMiksIGV4dHJhY3RlZCBieSBhIHBhcnNlcjogJycne3N1bW1hcmllcyhwYWlyLnByb2plY3QpW3BhaXIuY29kZV19JycnICIKICAgICAgICByZXR1cm4gbGMuU1lTVEVNX01FU1NBR0UsIHVzZXIKICAgIHJldHVybiB2YXJpYW50CgoKQ09NQklOQVRJT05TID0geyIrIi5qb2luKHAgZm9yIHAgaW4gUEFSVFMgaWYgcCBpbiBzdWJzZXQpOiBjb21iaW5lZChzdWJzZXQpCiAgICAgICAgICAgICAgICBmb3Igc3Vic2V0IGluICh7cCBmb3IgaSwgcCBpbiBlbnVtZXJhdGUoUEFSVFMpIGlmIG1hc2sgPj4gaSAmIDF9IGZvciBtYXNrIGluIHJhbmdlKDE2KSkKICAgICAgICAgICAgICAgIGlmIGxlbihzdWJzZXQpID49IDJ9ClZBUklBTlRTID0geyoqQkFTRV9WQVJJQU5UUywgKipDT01CSU5BVElPTlN9CgoKZGVmIGZpbGVzX3VzZWQodmFyaWFudCwgcHJvamVjdHMpOgogICAgIiIiVGhlIHByb21wdCBhbmQgZGF0YSBmaWxlcyBhIHZhcmlhbnQgcmVhZHMsIGZvciB0aGUgcHJvdmVuYW5jZSByZWNvcmQgb2YgYSBydW4uIiIiCiAgICBwZXJfdmFyaWFudCA9IHsKICAgICAgICAiYmFzZWxpbmUiOiBbXSwKICAgICAgICAiZGVmaW5pdGlvbiI6IFsicHJvbXB0cy9kZWZpbml0aW9uLnR4dCJdLAogICAgICAgICJzdW1tYXJ5IjogW2YiZGF0YS9zdW1tYXJpZXMte3B9Lmpzb24iIGZvciBwIGluIHByb2plY3RzXSwKICAgICAgICAicHJvamVjdCI6IFtmInByb21wdHMvcHJvamVjdC17cH0udHh0IiBmb3IgcCBpbiBwcm9qZWN0c10sCiAgICAgICAgImV4YW1wbGVzIjogW2YicHJvbXB0cy9leGFtcGxlcy17cH0udHh0IiBmb3IgcCBpbiBwcm9qZWN0c10sCiAgICB9CiAgICByZXR1cm4gW2YgZm9yIHBhcnQgaW4gdmFyaWFudC5zcGxpdCgiKyIpIGZvciBmIGluIHBlcl92YXJpYW50W3BhcnRdXQo=' | base64 -d > LiSSA-Replication-Package/harness/variants.py
echo 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJQcmVwYXJlcyB0aGUgaW5wdXRzIG9mIHRoZSBSUTIgdmFyaWFudHMgYW5kIGNoZWNrcyB0aGF0IGV2ZXJ5IHByb21wdCBmaXRzIGludG8gdGhlIGNvbnRleHQgd2luZG93LgoKMS4gQ29kZSBzdW1tYXJpZXM6IGRhdGEvc3VtbWFyaWVzLTxwcm9qZWN0Pi5qc29uLCB7Y29kZSBmaWxlOiBzdW1tYXJ5fSwgZnJvbSBqYXZhcGFyc2UucHkgKGRldGVybWluaXN0aWMpLgoyLiBDcm9zcy1wcm9qZWN0IGV4YW1wbGVzOiBwcm9tcHRzL2V4YW1wbGVzLTxwcm9qZWN0Pi50eHQsIHdyaXR0ZW4gb25jZSBhbmQgdGhlbiBmcm96ZW4gKC0tZm9yY2UtZXhhbXBsZXMgcmV3cml0ZXMpLgogICBGb3IgYSB0YXJnZXQgcHJvamVjdCwgb25lIGxpbmtlZCBwYWlyIGZyb20gdGhlIGZpcnN0IG90aGVyIHByb2plY3QgYW5kIG9uZSByZXRyaWV2ZWQtYnV0LXVubGlua2VkIHBhaXIgZnJvbSB0aGUKICAgc2Vjb25kLCBkcmF3biB3aXRoIGEgZml4ZWQgc2VlZCBmcm9tIHRob3NlIHByb2plY3RzJyBkZXYgc2xpY2VzLCBzbyBubyB0ZXN0LXNsaWNlIHJlcXVpcmVtZW50IGV2ZXIgYXBwZWFycyBpbiBhCiAgIHByb21wdC4gT25seSBzaG9ydCBhcnRpZmFjdHMgcXVhbGlmeSwgc28gdGhlIGV4YW1wbGVzIGFkZCBsaXR0bGUgdG8gdGhlIHByb21wdC4gZGF0YS9leGFtcGxlcy5qc29uIHJlY29yZHMgdGhlIHBhaXJzLgozLiBMZW5ndGggY2hlY2s6IHRoZSBsb25nZXN0IHByb21wdCBvZiBlYWNoIHZhcmlhbnQgb3ZlciBhbGwgY2FuZGlkYXRlIHBhaXJzLCBpbiBlc3RpbWF0ZWQgdG9rZW5zLgoKICAgIHB5dGhvbjMgaGFybmVzcy9wcmVwYXJlX3Byb21wdHMucHkgWy0tY2FuZGlkYXRlcyBkYXRhL2NhbmRpZGF0ZXMtcnExLmpzb25dIFstLWZvcmNlLWV4YW1wbGVzXQoiIiIKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgcmFuZG9tCmltcG9ydCBzeXMKCmltcG9ydCBldmFsdWF0ZQppbXBvcnQgamF2YXBhcnNlCmltcG9ydCBsaXNzYV9jb21wYXQgYXMgbGMKaW1wb3J0IHZhcmlhbnRzCgpNQVhfUkVRVUlSRU1FTlRfQ0hBUlMgPSAxMDAwCk1BWF9DT0RFX0NIQVJTID0gMTUwMApDSEFSU19QRVJfVE9LRU4gPSAzLjUgICMgbWVhc3VyZWQgNC4wLTUuMCBvbiB0aGUgcGlsb3QgKHByb21wdF9ldmFsX2NvdW50KTsgMy41IGxlYXZlcyBhIG1hcmdpbgpBTlNXRVJfUkVTRVJWRSA9IDEwMjQgICMgdG9rZW5zIGtlcHQgZnJlZSBmb3IgdGhlIGFuc3dlciAoYXMgaW4gUlExJ3MgcHJlZmxpZ2h0KQoKCmRlZiB3cml0ZV9zdW1tYXJpZXMoKToKICAgIGZvciBwcm9qZWN0IGluIGxjLlBST0pFQ1RTOgogICAgICAgIGNvZGUgPSBsYy5sb2FkX2FydGlmYWN0cyhsYy5EQVRBU0VUUyAvIHByb2plY3QgLyAiQ0MiKQogICAgICAgIHJlc3VsdCA9IHtuYW1lOiBqYXZhcGFyc2Uuc3VtbWFyeShqYXZhcGFyc2UucGFyc2UodGV4dCkpIGZvciBuYW1lLCB0ZXh0IGluIGNvZGUuaXRlbXMoKX0KICAgICAgICAodmFyaWFudHMuREFUQSAvIGYic3VtbWFyaWVzLXtwcm9qZWN0fS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTEsIGVuc3VyZV9hc2NpaT1GYWxzZSkgKyAiXG4iKQogICAgICAgIGVtcHR5ID0gW25hbWUgZm9yIG5hbWUsIHRleHQgaW4gcmVzdWx0Lml0ZW1zKCkgaWYgdGV4dC5zdGFydHN3aXRoKCIobm8gSmF2YSIpXQogICAgICAgIHByaW50KGYie3Byb2plY3R9OiB7bGVuKHJlc3VsdCl9IHN1bW1hcmllczsgd2l0aG91dCBKYXZhIGRlY2xhcmF0aW9uczoge2VtcHR5IG9yICdub25lJ30iKQoKCmRlZiBjaG9vc2VfZXhhbXBsZXMoY2FuZGlkYXRlcywgc3BsaXQpOgogICAgY2hvc2VuID0ge30KICAgIGZvciB0YXJnZXQgaW4gbGMuUFJPSkVDVFM6CiAgICAgICAgb3RoZXJzID0gW3AgZm9yIHAgaW4gbGMuUFJPSkVDVFMgaWYgcCAhPSB0YXJnZXRdCiAgICAgICAgY2hvc2VuW3RhcmdldF0gPSBbXQogICAgICAgIGZvciBzb3VyY2UsIGxpbmtlZCBpbiAoKG90aGVyc1swXSwgVHJ1ZSksIChvdGhlcnNbMV0sIEZhbHNlKSk6CiAgICAgICAgICAgIHJlcXVpcmVtZW50cyA9IGxjLmxvYWRfYXJ0aWZhY3RzKGxjLkRBVEFTRVRTIC8gc291cmNlIC8gIlVDIikKICAgICAgICAgICAgY29kZSA9IGxjLmxvYWRfYXJ0aWZhY3RzKGxjLkRBVEFTRVRTIC8gc291cmNlIC8gIkNDIikKICAgICAgICAgICAgZ29sZCwgZGV2ID0gbGMubG9hZF9nb2xkKHNvdXJjZSksIHNldChzcGxpdFsicHJvamVjdHMiXVtzb3VyY2VdWyJkZXYiXSkKICAgICAgICAgICAgcG9vbCA9IFsocmVxLCB0YXJnZXRfZmlsZSkgZm9yIHJlcSBpbiByZXF1aXJlbWVudHMgaWYgbGMubGlua19pZChyZXEpIGluIGRldgogICAgICAgICAgICAgICAgICAgIGZvciB0YXJnZXRfZmlsZSwgXyBpbiBjYW5kaWRhdGVzWyJwcm9qZWN0cyJdW3NvdXJjZV1bcmVxXQogICAgICAgICAgICAgICAgICAgIGlmICgobGMubGlua19pZChyZXEpLCBsYy5saW5rX2lkKHRhcmdldF9maWxlKSkgaW4gZ29sZCkgPT0gbGlua2VkCiAgICAgICAgICAgICAgICAgICAgYW5kIGxlbihyZXF1aXJlbWVudHNbcmVxXSkgPD0gTUFYX1JFUVVJUkVNRU5UX0NIQVJTIGFuZCBsZW4oY29kZVt0YXJnZXRfZmlsZV0pIDw9IE1BWF9DT0RFX0NIQVJTXQogICAgICAgICAgICBpZiBub3QgcG9vbDoKICAgICAgICAgICAgICAgIHN5cy5leGl0KGYiTm8geydsaW5rZWQnIGlmIGxpbmtlZCBlbHNlICd1bmxpbmtlZCd9IGV4YW1wbGUgaW4ge3NvdXJjZX0gd2l0aGluIHRoZSBsZW5ndGggbGltaXRzLiIpCiAgICAgICAgICAgIHJlcSwgdGFyZ2V0X2ZpbGUgPSByYW5kb20uUmFuZG9tKGYiZXhhbXBsZXMte3RhcmdldH0te3NvdXJjZX0iKS5jaG9pY2UocG9vbCkKICAgICAgICAgICAgY2hvc2VuW3RhcmdldF0uYXBwZW5kKHsicHJvamVjdCI6IHNvdXJjZSwgInJlcXVpcmVtZW50IjogcmVxLCAiY29kZSI6IHRhcmdldF9maWxlLCAibGlua2VkIjogbGlua2VkLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJyZXF1aXJlbWVudF90ZXh0IjogcmVxdWlyZW1lbnRzW3JlcV0sICJjb2RlX3RleHQiOiBjb2RlW3RhcmdldF9maWxlXX0pCiAgICByZXR1cm4gY2hvc2VuCgoKZGVmIHdyaXRlX2V4YW1wbGVzKGNob3NlbiwgZm9yY2UpOgogICAgcmVjb3JkID0gdmFyaWFudHMuREFUQSAvICJleGFtcGxlcy5qc29uIgogICAgZm9yIHRhcmdldCwgZXhhbXBsZXMgaW4gY2hvc2VuLml0ZW1zKCk6CiAgICAgICAgcGF0aCA9IHZhcmlhbnRzLlBST01QVFMgLyBmImV4YW1wbGVzLXt0YXJnZXR9LnR4dCIKICAgICAgICBpZiBwYXRoLmV4aXN0cygpIGFuZCBub3QgZm9yY2U6CiAgICAgICAgICAgIHByaW50KGYie3BhdGgubmFtZX0gZXhpc3RzIGFuZCBpcyBmcm96ZW4gKHVzZSAtLWZvcmNlLWV4YW1wbGVzIHRvIHJld3JpdGUgaXQpIikKICAgICAgICAgICAgY29udGludWUKICAgICAgICBibG9ja3MgPSBbZiJFeGFtcGxlIHtpfTpcblJlcXVpcmVtZW50OiAnJyd7ZVsncmVxdWlyZW1lbnRfdGV4dCddfScnJ1xuU291cmNlIGNvZGU6ICcnJ3tlWydjb2RlX3RleHQnXX0nJydcbiIKICAgICAgICAgICAgICAgICAgZiJBbnN3ZXI6IDx0cmFjZT57J3llcycgaWYgZVsnbGlua2VkJ10gZWxzZSAnbm8nfTwvdHJhY2U+IiBmb3IgaSwgZSBpbiBlbnVtZXJhdGUoZXhhbXBsZXMsIDEpXQogICAgICAgIHBhdGgud3JpdGVfdGV4dCgiXG4iLmpvaW4oYmxvY2tzKSArICJcbiIsIGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgcHJpbnQoZiJXcm90ZSB7cGF0aC5uYW1lfTogIiArICI7ICIuam9pbihmIntlWydwcm9qZWN0J119IHtlWydyZXF1aXJlbWVudCddfSAvIHtlWydjb2RlJ119ICIKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGYiKHsnbGlua2VkJyBpZiBlWydsaW5rZWQnXSBlbHNlICdub3QgbGlua2VkJ30pIiBmb3IgZSBpbiBleGFtcGxlcykpCiAgICAgICAgZXhpc3RpbmcgPSBqc29uLmxvYWRzKHJlY29yZC5yZWFkX3RleHQoKSkgaWYgcmVjb3JkLmV4aXN0cygpIGVsc2Uge30KICAgICAgICBleGlzdGluZ1t0YXJnZXRdID0gW3trOiBlW2tdIGZvciBrIGluICgicHJvamVjdCIsICJyZXF1aXJlbWVudCIsICJjb2RlIiwgImxpbmtlZCIpfSBmb3IgZSBpbiBleGFtcGxlc10KICAgICAgICByZWNvcmQud3JpdGVfdGV4dChqc29uLmR1bXBzKGV4aXN0aW5nLCBpbmRlbnQ9MSkgKyAiXG4iKQoKCmRlZiBsZW5ndGhfY2hlY2soY2FuZGlkYXRlcyk6CiAgICBsaW1pdCA9IGxjLk5VTV9DVFggLSBBTlNXRVJfUkVTRVJWRQogICAgcHJpbnQoZiJcbkxvbmdlc3QgcHJvbXB0IHBlciB2YXJpYW50IChlc3RpbWF0ZWQgYXQge0NIQVJTX1BFUl9UT0tFTn0gY2hhcmFjdGVycyBwZXIgdG9rZW47IGxpbWl0IHtsaW1pdH0gdG9rZW5zKToiKQogICAgb2sgPSBUcnVlCiAgICBmb3IgbmFtZSwgdmFyaWFudCBpbiB2YXJpYW50cy5CQVNFX1ZBUklBTlRTLml0ZW1zKCk6CiAgICAgICAgbG9uZ2VzdCwgd2hlcmUgPSAwLCBOb25lCiAgICAgICAgZm9yIHByb2plY3QgaW4gbGMuUFJPSkVDVFM6CiAgICAgICAgICAgIHJlcXVpcmVtZW50cyA9IGxjLmxvYWRfYXJ0aWZhY3RzKGxjLkRBVEFTRVRTIC8gcHJvamVjdCAvICJVQyIpCiAgICAgICAgICAgIGNvZGUgPSBsYy5sb2FkX2FydGlmYWN0cyhsYy5EQVRBU0VUUyAvIHByb2plY3QgLyAiQ0MiKQogICAgICAgICAgICBmb3IgcmVxLCByYW5rZWQgaW4gY2FuZGlkYXRlc1sicHJvamVjdHMiXVtwcm9qZWN0XS5pdGVtcygpOgogICAgICAgICAgICAgICAgZm9yIHRhcmdldCwgXyBpbiByYW5rZWQ6CiAgICAgICAgICAgICAgICAgICAgc3lzdGVtLCB1c2VyID0gdmFyaWFudCh2YXJpYW50cy5QYWlyKHByb2plY3QsIHJlcSwgdGFyZ2V0LCByZXF1aXJlbWVudHNbcmVxXSwgY29kZVt0YXJnZXRdKSkKICAgICAgICAgICAgICAgICAgICBpZiBsZW4oc3lzdGVtKSArIGxlbih1c2VyKSA+IGxvbmdlc3Q6CiAgICAgICAgICAgICAgICAgICAgICAgIGxvbmdlc3QsIHdoZXJlID0gbGVuKHN5c3RlbSkgKyBsZW4odXNlciksIGYie3Byb2plY3R9IHtyZXF9IC8ge3RhcmdldH0iCiAgICAgICAgdG9rZW5zID0gcm91bmQobG9uZ2VzdCAvIENIQVJTX1BFUl9UT0tFTikKICAgICAgICBvayAmPSB0b2tlbnMgPD0gbGltaXQKICAgICAgICBwcmludChmIiAge25hbWU6MTF9IHtsb25nZXN0OjZ9IGNoYXJhY3RlcnMsIGFib3V0IHt0b2tlbnM6NX0gdG9rZW5zICh7d2hlcmV9KXsnJyBpZiB0b2tlbnMgPD0gbGltaXQgZWxzZSAnICBUT08gTE9ORyd9IikKICAgIHJldHVybiBvawoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fLCBmb3JtYXR0ZXJfY2xhc3M9YXJncGFyc2UuUmF3RGVzY3JpcHRpb25IZWxwRm9ybWF0dGVyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jYW5kaWRhdGVzIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJkZWZhdWx0OiBkYXRhL2NhbmRpZGF0ZXMtcnExLmpzb24sIGVsc2UgLWxvY2FsIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZm9yY2UtZXhhbXBsZXMiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIGNhbmRpZGF0ZXNfZmlsZSA9IGxjLnBhdGhsaWIuUGF0aChhcmdzLmNhbmRpZGF0ZXMpIGlmIGFyZ3MuY2FuZGlkYXRlcyBlbHNlICgKICAgICAgICB2YXJpYW50cy5EQVRBIC8gImNhbmRpZGF0ZXMtcnExLmpzb24iIGlmICh2YXJpYW50cy5EQVRBIC8gImNhbmRpZGF0ZXMtcnExLmpzb24iKS5leGlzdHMoKQogICAgICAgIGVsc2UgdmFyaWFudHMuREFUQSAvICJjYW5kaWRhdGVzLWxvY2FsLmpzb24iKQogICAgY2FuZGlkYXRlcyA9IGpzb24ubG9hZHMoY2FuZGlkYXRlc19maWxlLnJlYWRfdGV4dCgpKQogICAgc3BsaXQgPSBqc29uLmxvYWRzKGV2YWx1YXRlLlNQTElUX0ZJTEUucmVhZF90ZXh0KCkpCiAgICBwcmludChmIkNhbmRpZGF0ZXM6IHtjYW5kaWRhdGVzX2ZpbGUubmFtZX0iKQogICAgd3JpdGVfc3VtbWFyaWVzKCkKICAgIHdyaXRlX2V4YW1wbGVzKGNob29zZV9leGFtcGxlcyhjYW5kaWRhdGVzLCBzcGxpdCksIGFyZ3MuZm9yY2VfZXhhbXBsZXMpCiAgICBpZiBub3QgbGVuZ3RoX2NoZWNrKGNhbmRpZGF0ZXMpOgogICAgICAgIHN5cy5leGl0KCJTb21lIHByb21wdHMgd291bGQgbm90IGZpdCBpbnRvIHRoZSBjb250ZXh0IHdpbmRvdzsgc2hvcnRlbiB0aGUgYWRkZWQgaW5mb3JtYXRpb24uIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==' | base64 -d > LiSSA-Replication-Package/harness/prepare_prompts.py
echo 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJTY29yZXMgcHJlZGljdGVkIGxpbmtzIG9uIGEgc2xpY2Ugb2YgdGhlIHJlcXVpcmVtZW50cywgd2l0aCBMaVNTQSdzIGZvcm11bGFzLgoKUGVyIHByb2plY3Q6IHByZWNpc2lvbiwgcmVjYWxsLCBGMSwgRjIgb3ZlciB0aGUgbGlua3Mgd2hvc2UgcmVxdWlyZW1lbnQgaXMgaW4gdGhlIHNsaWNlOyB0aGVuIHRoZSBwbGFpbiBhdmVyYWdlIG92ZXIKdGhlIHByb2plY3RzIGFuZCB0aGUgYXZlcmFnZSB3ZWlnaHRlZCBieSBnb2xkIGxpbmtzIChhcyBMaVNTQSByZXBvcnRzIHRoZW0pLiBQZXIgcmVxdWlyZW1lbnQ6IFRQLCBGUCwgRk4gYW5kIEYxIGZvcgp0aGUgV2lsY294b24gdGVzdHMuIEYxIGlzIGxlZnQgdW5kZWZpbmVkIChOYU4pIGZvciBhIHJlcXVpcmVtZW50IHdpdGggbm8gZ29sZCBsaW5rIGFuZCBubyBwcmVkaWN0ZWQgbGluazsgaG93IHN1Y2gKcmVxdWlyZW1lbnRzIGVudGVyIHRoZSB0ZXN0cyBpcyBkZWNpZGVkIGluIHRoZSBhbmFseXNpcywgbm90IGhlcmUuCgpTY29yZSBMaVNTQSdzIG93biBSUTEgb3V0cHV0IG9uIGEgc2xpY2U6CiAgICBweXRob24zIGhhcm5lc3MvZXZhbHVhdGUucHkgLS1zbGljZSB0ZXN0IC0tbGlua3MgU01PUz1wYXRoL3RyYWNlTGlua3MtU01PU18uLi5jc3YgZVRvdXJfZW49Li4uIGlUcnVzdD0uLi4KIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHBhdGhsaWIKCmltcG9ydCBsaXNzYV9jb21wYXQgYXMgbGMKClNQTElUX0ZJTEUgPSBsYy5IQVJORVNTIC8gImRhdGEiIC8gInNwbGl0Lmpzb24iCgoKZGVmIHJlcXVpcmVtZW50c19pbihzbGljZV9uYW1lLCBwcm9qZWN0LCBzcGxpdD1Ob25lKToKICAgIGlmIHNsaWNlX25hbWUgPT0gImFsbCI6CiAgICAgICAgcmV0dXJuIHtsYy5saW5rX2lkKG5hbWUpIGZvciBuYW1lIGluIGxjLmxvYWRfYXJ0aWZhY3RzKGxjLkRBVEFTRVRTIC8gcHJvamVjdCAvICJVQyIpfQogICAgc3BsaXQgPSBzcGxpdCBvciBqc29uLmxvYWRzKFNQTElUX0ZJTEUucmVhZF90ZXh0KCkpCiAgICByZXR1cm4gc2V0KHNwbGl0WyJwcm9qZWN0cyJdW3Byb2plY3RdW3NsaWNlX25hbWVdKQoKCmRlZiBldmFsdWF0ZShwcmVkaWN0ZWQsIHNsaWNlX25hbWUsIHNwbGl0PU5vbmUpOgogICAgIiIicHJlZGljdGVkOiB7cHJvamVjdDogeyhyZXF1aXJlbWVudCBpZCwgY29kZSBpZCksIC4uLn19IC0+IHsicHJvamVjdHMiOiBbLi4uXSwgInJlcXVpcmVtZW50cyI6IFsuLi5dfSIiIgogICAgcHJvamVjdHMsIHBlcl9yZXF1aXJlbWVudCA9IFtdLCBbXQogICAgZm9yIHByb2plY3QsIGxpbmtzIGluIHByZWRpY3RlZC5pdGVtcygpOgogICAgICAgIHJlcXMgPSByZXF1aXJlbWVudHNfaW4oc2xpY2VfbmFtZSwgcHJvamVjdCwgc3BsaXQpCiAgICAgICAgZ29sZCA9IHtsaW5rIGZvciBsaW5rIGluIGxjLmxvYWRfZ29sZChwcm9qZWN0KSBpZiBsaW5rWzBdIGluIHJlcXN9CiAgICAgICAgbGlua3MgPSB7bGluayBmb3IgbGluayBpbiBsaW5rcyBpZiBsaW5rWzBdIGluIHJlcXN9CiAgICAgICAgdHAsIGZwLCBmbiA9IGxjLmNvdW50cyhsaW5rcywgZ29sZCkKICAgICAgICBwcm9qZWN0cy5hcHBlbmQoeyJwcm9qZWN0IjogcHJvamVjdCwgInJlcXVpcmVtZW50cyI6IGxlbihyZXFzKSwgImdvbGQiOiBsZW4oZ29sZCksICJwcmVkaWN0ZWQiOiBsZW4obGlua3MpLAogICAgICAgICAgICAgICAgICAgICAgICAgInRwIjogdHAsICJmcCI6IGZwLCAiZm4iOiBmbiwgKipsYy5zY29yZXModHAsIGZwLCBmbil9KQogICAgICAgIGZvciByZXEgaW4gc29ydGVkKHJlcXMpOgogICAgICAgICAgICByX3RwLCByX2ZwLCByX2ZuID0gbGMuY291bnRzKHtsIGZvciBsIGluIGxpbmtzIGlmIGxbMF0gPT0gcmVxfSwge2wgZm9yIGwgaW4gZ29sZCBpZiBsWzBdID09IHJlcX0pCiAgICAgICAgICAgIGYxID0gMiAqIHJfdHAgLyAoMiAqIHJfdHAgKyByX2ZwICsgcl9mbikgaWYgcl90cCArIHJfZnAgKyByX2ZuIGVsc2UgbWF0aC5uYW4KICAgICAgICAgICAgcGVyX3JlcXVpcmVtZW50LmFwcGVuZCh7InByb2plY3QiOiBwcm9qZWN0LCAicmVxdWlyZW1lbnQiOiByZXEsICJ0cCI6IHJfdHAsICJmcCI6IHJfZnAsICJmbiI6IHJfZm4sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJmMSI6IGYxfSkKICAgIGlmIGxlbihwcm9qZWN0cykgPiAxOgogICAgICAgIHJvd3MgPSBsaXN0KHByb2plY3RzKQogICAgICAgIGxpbmtzX3RvdGFsID0gc3VtKHJvd1siZ29sZCJdIGZvciByb3cgaW4gcm93cykKICAgICAgICBmb3IgbGFiZWwsIHdlaWdodCBpbiAoKCJBdmcuIiwgbGFtYmRhIHJvdzogMSAvIGxlbihyb3dzKSksICgidy4gQXZnLiIsIGxhbWJkYSByb3c6IHJvd1siZ29sZCJdIC8gbGlua3NfdG90YWwpKToKICAgICAgICAgICAgYXZlcmFnZWQgPSB7bTogc3VtKHdlaWdodChyb3cpICogcm93W21dIGZvciByb3cgaW4gcm93cykgZm9yIG0gaW4gKCJwIiwgInIiLCAiZjEiLCAiZjIiKX0KICAgICAgICAgICAgcHJvamVjdHMuYXBwZW5kKHsicHJvamVjdCI6IGxhYmVsLCAqKmF2ZXJhZ2VkfSkKICAgIHJldHVybiB7InNsaWNlIjogc2xpY2VfbmFtZSwgInByb2plY3RzIjogcHJvamVjdHMsICJyZXF1aXJlbWVudHMiOiBwZXJfcmVxdWlyZW1lbnR9CgoKZGVmIHRhYmxlKHJlc3VsdCk6CiAgICBsaW5lcyA9IFsifCBQcm9qZWN0IHwgUmVxcyB8IEdvbGQgfCBQcmVkLiB8IFRQIHwgRlAgfCBGTiB8IFAgfCBSIHwgRjEgfCBGMiB8IiwgInwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfC0tLXwtLS18LS0tfCJdCiAgICBmb3Igcm93IGluIHJlc3VsdFsicHJvamVjdHMiXToKICAgICAgICBuID0gW3N0cihyb3cuZ2V0KGssICIiKSkgZm9yIGsgaW4gKCJyZXF1aXJlbWVudHMiLCAiZ29sZCIsICJwcmVkaWN0ZWQiLCAidHAiLCAiZnAiLCAiZm4iKV0KICAgICAgICBsaW5lcy5hcHBlbmQoZiJ8IHtyb3dbJ3Byb2plY3QnXX0gfCAiICsgIiB8ICIuam9pbihuKSArICIgfCAiICsKICAgICAgICAgICAgICAgICAgICAgIiB8ICIuam9pbihmIntyb3dbbV06LjNmfSIgZm9yIG0gaW4gKCJwIiwgInIiLCAiZjEiLCAiZjIiKSkgKyAiIHwiKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2xpY2UiLCBjaG9pY2VzPVsiZGV2IiwgInRlc3QiLCAiYWxsIl0sIGRlZmF1bHQ9InRlc3QiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1saW5rcyIsIG5hcmdzPSIrIiwgcmVxdWlyZWQ9VHJ1ZSwgbWV0YXZhcj0iUFJPSkVDVD10cmFjZUxpbmtzLmNzdiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWpzb24iLCBoZWxwPSJhbHNvIHdyaXRlIHRoZSBmdWxsIHJlc3VsdCAod2l0aCBwZXItcmVxdWlyZW1lbnQgcm93cykgdG8gdGhpcyBmaWxlIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBwcmVkaWN0ZWQgPSB7fQogICAgZm9yIGl0ZW0gaW4gYXJncy5saW5rczoKICAgICAgICBwcm9qZWN0LCBwYXRoID0gaXRlbS5zcGxpdCgiPSIsIDEpCiAgICAgICAgcHJlZGljdGVkW3Byb2plY3RdID0gbGMucmVhZF90cmFjZV9saW5rcyhwYXRoKQogICAgcmVzdWx0ID0gZXZhbHVhdGUocHJlZGljdGVkLCBhcmdzLnNsaWNlKQogICAgcHJpbnQodGFibGUocmVzdWx0KSkKICAgIGlmIGFyZ3MuanNvbjoKICAgICAgICBwYXRobGliLlBhdGgoYXJncy5qc29uKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MSkgKyAiXG4iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK' | base64 -d > LiSSA-Replication-Package/harness/evaluate.py
echo 'IiIiUm91bmQgMiBjaG9pY2UgYW5kIHRoZSBkZXYgc3VtbWFyeSBmb3IgdGhlIFJRMi9SUTMgcGlwZWxpbmUgb24gQ29sYWIuCgogICAgcHl0aG9uMyAvY29udGVudC9ycTIzLnB5IGNob29zZSBNT0RFTF9UQUcgREVWSUNFICAgLT4gSlNPTiBsaW5lIHdpdGggdGhlIHJvdW5kIDIgY29tYmluYXRpb24KICAgIHB5dGhvbjMgL2NvbnRlbnQvcnEyMy5weSBzdW1tYXJ5IERFVklDRSAgICAgICAgICAgICAtPiBkZXYgcmVzdWx0cyBwZXIgbW9kZWwsIHdpdGggR1BULTRvJ3MgZGV2IEYxIGFzIHRoZSB0YXJnZXQKIiIiCmltcG9ydCBjc3YKaW1wb3J0IGdsb2IKaW1wb3J0IGpzb24KaW1wb3J0IHBhdGhsaWIKaW1wb3J0IHN5cwoKc3lzLnBhdGguaW5zZXJ0KDAsICJoYXJuZXNzIikKaW1wb3J0IGFuYWx5emUgICMgbm9xYTogRTQwMgppbXBvcnQgZXZhbHVhdGUgICMgbm9xYTogRTQwMgppbXBvcnQgbGlzc2FfY29tcGF0IGFzIGxjICAjIG5vcWE6IEU0MDIKaW1wb3J0IHZhcmlhbnRzICAjIG5vcWE6IEU0MDIKaW1wb3J0IHZlcmlmeSAgIyBub3FhOiBFNDAyCgpHUFQ0T19ERVYgPSAwLjM4NQpNT0RFTFMgPSAoInF3ZW4yLjUtY29kZXItMTRiIiwgImxsYW1hMy4xLThiIikKCgpkZWYgcnVucyh0YWcsIGRldmljZSk6CiAgICByZXR1cm4gc29ydGVkKGdsb2IuZ2xvYihmImhhcm5lc3MvcnVucy8qX197dGFnfV9fe2RldmljZX1fX2RldiIpKQoKCmRlZiBuYW1lX29mKHJ1bik6CiAgICByZXR1cm4gcGF0aGxpYi5QYXRoKHJ1bikubmFtZS5zcGxpdCgiX18iKVswXQoKCmRlZiBhdmdfZjEocnVuLCBwYXJzZXIpOgogICAgXywgbGlua3MsIHNwbGl0ID0gYW5hbHl6ZS5sb2FkKHJ1biwgcGFyc2VyKQogICAgcmV0dXJuIGFuYWx5emUuYXZlcmFnZV9mMShldmFsdWF0ZS5ldmFsdWF0ZShsaW5rcywgImp1ZGdlZCIsIHNwbGl0KSkKCgpkZWYgY2hvb3NlKHRhZywgZGV2aWNlKToKICAgIHNjb3JlcyA9IHtuYW1lX29mKHIpOiBhdmdfZjEociwgInN0cmljdCIpIGZvciByIGluIHJ1bnModGFnLCBkZXZpY2UpIGlmIG5hbWVfb2YocikgaW4gdmFyaWFudHMuQkFTRV9WQVJJQU5UU30KICAgIHNpbmdsZXMgPSBzb3J0ZWQoKG4gZm9yIG4gaW4gc2NvcmVzIGlmIG4gIT0gImJhc2VsaW5lIiksIGtleT1sYW1iZGEgbjogLXNjb3Jlc1tuXSkKICAgIGhlbHBmdWwgPSBbbiBmb3IgbiBpbiBzaW5nbGVzIGlmIHNjb3Jlc1tuXSA+IHNjb3Jlc1siYmFzZWxpbmUiXV0KICAgIHBhcnRzID0gaGVscGZ1bCBpZiBsZW4oaGVscGZ1bCkgPj0gMiBlbHNlIHNpbmdsZXNbOjJdCiAgICBjYW5kaWRhdGVzID0ganNvbi5sb2FkcyhwYXRobGliLlBhdGgoImhhcm5lc3MvZGF0YS9jYW5kaWRhdGVzLXJxMS5qc29uIikucmVhZF90ZXh0KCkpCiAgICB0ZXh0cyA9IHtwOiAobGMubG9hZF9hcnRpZmFjdHMobGMuREFUQVNFVFMgLyBwIC8gIlVDIiksIGxjLmxvYWRfYXJ0aWZhY3RzKGxjLkRBVEFTRVRTIC8gcCAvICJDQyIpKSBmb3IgcCBpbiBsYy5QUk9KRUNUU30KICAgIGxpbWl0ID0gbGMuTlVNX0NUWCAtIDEwMjQKCiAgICBkZWYgdG9rZW5zKGNob3Nlbik6CiAgICAgICAgdmFyaWFudCwgbG9uZ2VzdCA9IHZhcmlhbnRzLmNvbWJpbmVkKHNldChjaG9zZW4pKSwgMAogICAgICAgIGZvciBwIGluIGxjLlBST0pFQ1RTOgogICAgICAgICAgICByZXF1aXJlbWVudHMsIGNvZGUgPSB0ZXh0c1twXQogICAgICAgICAgICBmb3IgcmVxLCByYW5rZWQgaW4gY2FuZGlkYXRlc1sicHJvamVjdHMiXVtwXS5pdGVtcygpOgogICAgICAgICAgICAgICAgZm9yIHRhcmdldCwgXyBpbiByYW5rZWQ6CiAgICAgICAgICAgICAgICAgICAgc3lzdGVtLCB1c2VyID0gdmFyaWFudCh2YXJpYW50cy5QYWlyKHAsIHJlcSwgdGFyZ2V0LCByZXF1aXJlbWVudHNbcmVxXSwgY29kZVt0YXJnZXRdKSkKICAgICAgICAgICAgICAgICAgICBsb25nZXN0ID0gbWF4KGxvbmdlc3QsIGxlbihzeXN0ZW0pICsgbGVuKHVzZXIpKQogICAgICAgIHJldHVybiBsb25nZXN0IC8gNC4wCgogICAgd2hpbGUgbGVuKHBhcnRzKSA+PSAyIGFuZCB0b2tlbnMocGFydHMpID4gbGltaXQ6CiAgICAgICAgcGFydHMgPSBwYXJ0c1s6LTFdCiAgICBjb21iaW5hdGlvbiA9ICIrIi5qb2luKHAgZm9yIHAgaW4gdmFyaWFudHMuUEFSVFMgaWYgcCBpbiBwYXJ0cykgaWYgbGVuKHBhcnRzKSA+PSAyIGVsc2UgIiIKICAgIHByaW50KGpzb24uZHVtcHMoeyJtb2RlbCI6IHRhZywgImRldl9mMSI6IHNjb3JlcywgImhlbHBmdWwiOiBoZWxwZnVsLCAiY29tYmluYXRpb24iOiBjb21iaW5hdGlvbn0pKQoKCmRlZiB2ZXJpZmllZChydW4sIHBhcnNlcj0ic3RyaWN0Iik6CiAgICBvdXQgPSBwYXRobGliLlBhdGgocnVuKSAvIGYidmVyaWZ5LXtwYXJzZXJ9IgogICAgaWYgbm90IChvdXQgLyAibGlua3MuanNvbmwiKS5leGlzdHMoKToKICAgICAgICByZXR1cm4gTm9uZQogICAgcmVjb3JkcyA9IFtqc29uLmxvYWRzKGxpbmUpIGZvciBsaW5lIGluIChvdXQgLyAibGlua3MuanNvbmwiKS5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCkgaWYgbGluZV0KICAgIHdpdGggb3BlbihwYXRobGliLlBhdGgocnVuKSAvICJkZWNpc2lvbnMuY3N2IiwgbmV3bGluZT0iIikgYXMgZmlsZToKICAgICAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcihmaWxlKSkKICAgIHJldHVybiByb3dzLCByZWNvcmRzCgoKZGVmIGJ5X3RhdShyb3dzLCByZWNvcmRzLCBzdXBwb3J0ZWQsIHBhcnNlcj0ic3RyaWN0Iik6CiAgICByZXNjb3JlZCA9IFtdCiAgICBmb3IgcmVjb3JkIGluIHJlY29yZHM6CiAgICAgICAgbiA9IHN1bShjbGFpbVsidmVyZGljdCJdIGluIHN1cHBvcnRlZCBmb3IgY2xhaW0gaW4gcmVjb3JkWyJjbGFpbXMiXSkKICAgICAgICByZXNjb3JlZC5hcHBlbmQoeyoqcmVjb3JkLCAic2hhcmUiOiBuIC8gbGVuKHJlY29yZFsiY2xhaW1zIl0pIGlmIHJlY29yZFsiY2xhaW1zIl0gZWxzZSAwLjB9KQogICAgcmV0dXJuIHt0YXU6IGFuYWx5emUuYXZlcmFnZV9mMShyZXN1bHQpIGZvciB0YXUsIHJlc3VsdCBpbiB2ZXJpZnkubWV0cmljc19ieV90YXUocm93cywgcmVzY29yZWQsIHBhcnNlcikuaXRlbXMoKX0KCgpkZWYgc3VtbWFyeShkZXZpY2UpOgogICAgZm9yIHRhZyBpbiBNT0RFTFM6CiAgICAgICAgcHJpbnQoZiJcbiMjIHt0YWd9OiBkZXYgYXZlcmFnZSBGMSwgc3RyaWN0IHwgdG9sZXJhbnQgcGFyc2luZyIpCiAgICAgICAgYmVzdCA9ICgiIiwgLTEuMCkKICAgICAgICBmb3IgcnVuIGluIHJ1bnModGFnLCBkZXZpY2UpOgogICAgICAgICAgICBzdHJpY3QsIHRvbGVyYW50ID0gYXZnX2YxKHJ1biwgInN0cmljdCIpLCBhdmdfZjEocnVuLCAidG9sZXJhbnQiKQogICAgICAgICAgICBwcmludChmIiAge25hbWVfb2YocnVuKTo0MH0ge3N0cmljdDouM2Z9IHwge3RvbGVyYW50Oi4zZn0iKQogICAgICAgICAgICBiZXN0ID0gbWF4KGJlc3QsIChmIntuYW1lX29mKHJ1bil9IChubyB2ZXJpZmljYXRpb24pIiwgc3RyaWN0KSwga2V5PWxhbWJkYSB4OiB4WzFdKQogICAgICAgICAgICB2ID0gdmVyaWZpZWQocnVuKQogICAgICAgICAgICBpZiB2OgogICAgICAgICAgICAgICAgZm9yIGxhYmVsLCBzdXBwb3J0ZWQgaW4gKCgidW5jbGVhciA9IG5vdCBzdXBwb3J0ZWQiLCB7InN1cHBvcnRlZCJ9KSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAoInVuY2xlYXIgPSBzdXBwb3J0ZWQgKHNlbnNpdGl2aXR5KSIsIHsic3VwcG9ydGVkIiwgInVuY2xlYXIifSkpOgogICAgICAgICAgICAgICAgICAgIGYxID0gYnlfdGF1KCp2LCBzdXBwb3J0ZWQpCiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiIgICAgIHZlcmlmaWVkLCB7bGFiZWx9OiAiICsgIiAgIi5qb2luKGYidGF1IHt0fToge3g6LjNmfSIgZm9yIHQsIHggaW4gZjEuaXRlbXMoKSkpCiAgICAgICAgICAgICAgICAgICAgaWYgc3VwcG9ydGVkID09IHsic3VwcG9ydGVkIn06CiAgICAgICAgICAgICAgICAgICAgICAgIGZvciB0YXUsIHggaW4gZjEuaXRlbXMoKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIHRhdSAhPSAibm9uZSI6CiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYmVzdCA9IG1heChiZXN0LCAoZiJ7bmFtZV9vZihydW4pfSArIHZlcmlmaWNhdGlvbiwgdGF1IHt0YXV9IiwgeCksIGtleT1sYW1iZGEgeTogeVsxXSkKICAgICAgICB2ZXJkaWN0ID0gIkJFQVRTIEdQVC00byIgaWYgYmVzdFsxXSA+IEdQVDRPX0RFViBlbHNlICJiZWxvdyBHUFQtNG8iCiAgICAgICAgcHJpbnQoZiIgIEJFU1Qgb24gZGV2OiB7YmVzdFswXX0gPSB7YmVzdFsxXTouM2Z9IHZzIEdQVC00byB7R1BUNE9fREVWOi4zZn0gLT4ge3ZlcmRpY3R9IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgeyJjaG9vc2UiOiBsYW1iZGE6IGNob29zZShzeXMuYXJndlsyXSwgc3lzLmFyZ3ZbM10pLCAic3VtbWFyeSI6IGxhbWJkYTogc3VtbWFyeShzeXMuYXJndlsyXSl9W3N5cy5hcmd2WzFdXSgpCg==' | base64 -d > /content/rq23.py
pip -q install -r LiSSA-Replication-Package/harness/requirements.txt
(cd LiSSA-Replication-Package && sha256sum harness/variants.py harness/prepare_prompts.py harness/evaluate.py | cut -c1-12)
/usr/lib/jvm/java-21-openjdk-amd64/bin/java -version 2>&1 | grep -i version; mvn -v 2>/dev/null | grep -i "apache maven"; ollama -v 2>&1 | tail -1

69580753cedd
0037b645a15c
d2a54b44634c
openjdk version "21.0.13.0.3" 2026-10-24
Apache Maven 3.8.7


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


**Cell 3: save to and restore from Google Drive** (it refuses to run when Drive is not connected)

In [3]:
%%bash
cat > /content/sync.sh <<'EOF'
REPO=/content/LiSSA-Replication-Package
SAVE=/content/drive/MyDrive/lissa-colab
SNAP=/content/drive/MyDrive/lissa-colab-snapshots
PATHS="LiSSA-RATLR-V2/lissa/cache-local-r2c LiSSA-RATLR-V2/lissa/results-local harness/cache harness/runs harness/data harness/prompts"
if ! mountpoint -q /content/drive; then echo "$(date -u +%FT%TZ) Google Drive is not connected: nothing saved or restored. Run cell 1."; exit 1; fi
case "$1" in
  save)
    for p in $PATHS; do [ -d "$REPO/$p" ] && mkdir -p "$SAVE/$p" && rsync -a "$REPO/$p/" "$SAVE/$p/"; done
    date -u +%FT%TZ > "$SAVE/last-save.txt" ;;
  restore)
    for p in $PATHS; do [ -d "$SAVE/$p" ] && mkdir -p "$REPO/$p" && rsync -a "$SAVE/$p/" "$REPO/$p/"; done
    echo "restored from Drive (last save: $(cat "$SAVE/last-save.txt" 2>/dev/null || echo never))" ;;
  snapshot)
    mkdir -p "$SNAP"; cd "$REPO" && tar czf "$SNAP/.tmp.tgz" $(for p in $PATHS; do [ -d "$p" ] && echo "$p"; done) \
      && mv "$SNAP/.tmp.tgz" "$SNAP/snapshot-$(date -u +%Y%m%d-%H%M).tgz"
    ls -1t "$SNAP"/snapshot-*.tgz 2>/dev/null | tail -n +7 | xargs -r rm -f ;;
esac
EOF
bash /content/sync.sh restore

restored from Drive (last save: 2026-11-15T14:15:29Z)


**Cell 4: start Ollama, save every 2 minutes, snapshot every hour**

In [4]:
import os, subprocess, time, urllib.request
os.chdir("/content/LiSSA-Replication-Package")
env = {**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_CONTEXT_LENGTH": "16384"}
ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
autosave = subprocess.Popen(["bash", "-c",
    "i=0; while true; do sleep 120; bash /content/sync.sh save >> /content/sync.log 2>&1; "
    "i=$((i+1)); [ $((i % 30)) -eq 0 ] && bash /content/sync.sh snapshot >> /content/sync.log 2>&1; done"])
RQ1, CACHE = "LiSSA-RATLR-V2/lissa/results-local/rq1", "LiSSA-RATLR-V2/lissa/cache-local-r2c"
time.sleep(5)
print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())

{"version":"0.40.2"}


**Pipeline (dev slice, passes 1 and 2).** The log is saved to Drive as `harness/runs/pipeline-dev.log`; it ends with `ROUND 2 DEV DONE`.

In [5]:
%%bash
set -eo pipefail
cd /content/LiSSA-Replication-Package
mkdir -p harness/runs/.markers
exec > >(tee -a harness/runs/pipeline-dev.log) 2>&1
RQ1=LiSSA-RATLR-V2/lissa/results-local/rq1
CACHE=LiSSA-RATLR-V2/lissa/cache-local-r2c
D=colab-l4
MODELS="qwen2.5-coder:14b llama3.1:8b"
say() { echo "=== $(date -u +%FT%TZ) $*"; }
done_() { [ -f harness/runs/.markers/$1 ]; }
mark() { touch harness/runs/.markers/$1; }
best() { python3 harness/analyze.py --parser strict variant harness/runs/*__$1__${D}__dev | tee /dev/stderr | sed -n 's/^Highest average F1 (strict parsing): //p'; }
verify_once() { [ -f harness/runs/$1/verify-strict/metrics.json ] || { say "RQ3 dev verify $1"; python3 harness/verify.py --run harness/runs/$1 --parser strict --device $D; }; }

for m in $MODELS; do
  for i in 1 2 3 4 5; do ollama pull $m > /dev/null 2>&1 && break; say "pull $m failed, retry $i"; sleep 30; done
done

if ! done_ prep; then
  say "check_rq1"
  python3 harness/check_rq1.py --rq1-run $RQ1 --lissa-cache $CACHE --reask 20 | tee /tmp/check.txt
  grep -q "ALL CHECKS PASSED" /tmp/check.txt || { say "check_rq1 FAILED, stopping"; exit 1; }
  say "prepare_prompts"
  python3 harness/prepare_prompts.py --force-examples
  python3 harness/selftest.py
  bash /content/sync.sh save
  mark prep
fi

for M in $MODELS; do
  T=${M/:/-}
  if ! done_ dev1-$T; then
    say "RQ2 dev $M baseline"
    python3 harness/run.py --variant baseline --model $M --slice dev --lissa-cache $CACHE --device $D
    for v in definition summary project examples; do
      say "RQ2 dev $M $v"
      python3 harness/run.py --variant $v --model $M --slice dev --device $D
    done
    bash /content/sync.sh save
    B=$(best $T)
    verify_once baseline__${T}__${D}__dev
    verify_once $B
    python3 harness/analyze.py --parser strict tau harness/runs/baseline__${T}__${D}__dev
    bash /content/sync.sh save
    mark dev1-$T
  fi
done
say "ROUND 1 DEV DONE"
python3 /content/rq23.py summary $D

for M in $MODELS; do
  T=${M/:/-}
  if ! done_ dev2-$T; then
    CHOICE=$(python3 /content/rq23.py choose $T $D | tail -1)
    echo "$CHOICE"
    COMBO=$(echo "$CHOICE" | python3 -c 'import json,sys; print(json.load(sys.stdin)["combination"])')
    if [ -n "$COMBO" ]; then
      say "RQ2 dev round 2 $M $COMBO"
      python3 harness/run.py --variant $COMBO --model $M --slice dev --device $D
      bash /content/sync.sh save
    fi
    B=$(best $T)
    verify_once $B
    bash /content/sync.sh save
    mark dev2-$T
  fi
done

say "SUMMARY after round 2, dev slice"
python3 /content/rq23.py summary $D
say "GPT-4o CoT on the dev slice (target)"
python3 - <<'EOF'
import pathlib, sys
sys.path.insert(0, "harness")
import evaluate
R = pathlib.Path("LiSSA-RATLR-V2/lissa/results-r2c")
links = {p: {tuple(l.split(",")) for l in next(R.glob(f"traceLinks-{p}_133742243_artifact_artifact_reasoning_gpt_gpt-4o-2024-05-13.json_*.csv")).read_text().splitlines() if l}
         for p in ("SMOS", "eTour_en", "iTrust")}
print(evaluate.table(evaluate.evaluate(links, "dev")))
EOF
mark dev2
bash /content/sync.sh save
bash /content/sync.sh snapshot
say "ROUND 2 DEV DONE"

Process is interrupted.


**Before stopping:** save, then use Runtime > Disconnect and delete runtime.

In [6]:
!sha256sum /content/drive/MyDrive/lissa_pipeline2.sh /content/drive/MyDrive/lissa_rq23v2.py /content/drive/MyDrive/lissa_start2.sh | cut -c1-12

06929e34ad74
a5980eaa19f1
=== 2026-11-15T14:39:42Z PIPELINE v2: fresh same-session baselines; classification for both models first, verification last
=== 2026-11-15T14:39:42Z pull qwen2.5-coder:14b failed, retry 1
=== 2026-11-15T14:40:12Z pull qwen2.5-coder:14b failed, retry 2
^C


In [ ]:
!bash /content/drive/MyDrive/lissa_start2.sh

d52c55258d51
ollama {"version":"0.40.2"}
autosave loop started
pipeline v2 started
=== 2026-11-15T14:42:13Z PIPELINE v2: fresh same-session baselines; classification for both models first, verification last
=== 2026-11-15T14:42:15Z kept RQ1's cached baseline for qwen2.5-coder-14b as rq1cache-baseline__qwen2.5-coder-14b__colab-l4__dev-rq1
=== 2026-11-15T14:42:15Z RQ2 dev qwen2.5-coder:14b baseline (fresh, this session)
[361/1020] eTour_en UC36.txt #1 ErrorMessage.java: yes (6.1 s; about 0.8 h left)
[543/1020] iTrust UC15E1.txt #3 ErrorList.java: yes (4.7 s; about 0.6 h left)
[915/1020] iTrust UC3S1.txt #15 ErrorList.java: no (no tag) (2.9 s; about 0.1 h left)
| Avg. |  |  |  |  |  |  | 0.435 | 0.157 | 0.208 | 0.173 |
| Avg. |  |  |  |  |  |  | 0.453 | 0.310 | 0.332 | 0.314 |
=== 2026-11-15T16:00:06Z RQ2 dev qwen2.5-coder:14b definition
[361/1020] eTour_en UC36.txt #1 ErrorMessage.java: no (no tag) (3.2 s; about 0.9 h left)
[543/1020] iTrust UC15E1.txt #3 ErrorList.java: no (5.1 s; about

In [ ]:
!bash /content/sync.sh save && bash /content/sync.sh snapshot && cat /content/drive/MyDrive/lissa-colab/last-save.txt